# STEP5 FPI 산출 및 민감도 분석 

## 분석 개요

STEP4에서 산출한 감성점수를 활용해 **프랜차이즈 경쟁압력(FPI)**를 계산합니다

| 단계      | 내용                         |
| ------- | -------------------------- |
| STEP5-1 | 독립 브랜드와 프랜차이즈 간 거리 계산      |
| STEP5-2 | 반경 기반 거리 가중 경쟁압력지수(FPI) 산출 |
| STEP5-3 | 계산된 FPI를 원본 데이터에 결합        |
| STEP5-4 | 회귀분석 기반 경쟁압력 민감도 분석        |


---
**입력 데이터**
- `biz_sentiment.csv` : 5,899개의 행, 17개의 열 / TF-IDF 기반 감성점수를 포함한 업체 정보

**출력 데이터**
- `biz_sentiment_with_fpi.csv` : 기존 입력 데이터에 계산된 fpi를 추가한 데이터(프랜차이즈 업체의 경우, 0으로 채움)

# 공통라이브러리 및 설정

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from geopy.distance import geodesic
from tqdm import tqdm

PATH_to_data = r"C:\Users\seonu\Documents\yelp-franchise-analysis\results"
PATH_to_save = r"C:\Users\seonu\Documents\yelp-franchise-analysis\results"

# 데이터 준비

In [5]:
### 데이터 불러오기
df = pd.read_csv(f"{PATH_to_data}/biz_sentiment.csv")

### 'is_franchise' 컬럼 기준으로 독립브랜드와 프랜차이즈 분리
indie_df = df[df['is_franchise'] == False].copy()
fran_df = df[df['is_franchise'] == True].copy()

print("1. 데이터를 불러오고 독립/프랜차이즈를 분리합니다...")
print(f"분석 대상 독립 브랜드: {len(indie_df)}개")
print(f"주변 프랜차이즈 브랜드: {len(fran_df)}개")

1. 데이터를 불러오고 독립/프랜차이즈를 분리합니다...
분석 대상 독립 브랜드: 4818개
주변 프랜차이즈 브랜드: 1081개


---
# STEP5-1 거리계산

In [6]:
print("2. 모든 매장 간의 물리적 거리 계산 중...")
indie_lat = np.radians(indie_df['latitude'].values)[:, np.newaxis]
indie_lon = np.radians(indie_df['longitude'].values)[:, np.newaxis]
fran_lat = np.radians(fran_df['latitude'].values)[np.newaxis, :]
fran_lon = np.radians(fran_df['longitude'].values)[np.newaxis, :]

    # 하버사인(Haversine) 공식을 활용한 고속 행렬 연산
dlat = indie_lat - fran_lat
dlon = indie_lon - fran_lon
a = np.sin(dlat / 2)**2 + np.cos(indie_lat) * np.cos(fran_lat) * np.sin(dlon / 2)**2
distances_m = 2 * np.arcsin(np.sqrt(a)) * 6371000

2. 모든 매장 간의 물리적 거리 계산 중...


---
# STEP5-2 FPI 산출

### FPI 산출 공식
$$
FPI = \sum \frac{(\text{별점} \times \ln(\text{리뷰수}+1) \times \text{감성점수})}{(\text{거리}+k)^2}
$$

### 보정 사항
거리를 미터($\text{m}$) 단위 그대로 적용할 경우 지수적 감쇄로 인해 영향력이 사실상 $0$에 수렴하는 현상이 발생함. 이를 방지하고자 거리를 킬로미터($\text{km}$) 단위로 변환하고 보정 상수 $k=1$을 반영하여 공간적 마찰 효과를 합리적으로 조정함.

In [7]:
    # 프랜차이즈 분자 점수 계산
fran_scores = fran_df['stars'].values * np.log(fran_df['review_count'].values + 1) * fran_df['tfidf_sentiment'].values

print("3. 반경별 FPI 계산 및 변수 결합 진행 중...")
radii = [100, 200, 300, 500]
k = 1 # 거리 보정 상수

for r in radii:
    mask = distances_m <= r
    # 분모에 들어갈 거리를 km 단위로 변환 (예: 200m -> 0.2km)하여 영향력 보정
    distances_km = distances_m / 1000.0 
    
    fpi_terms = fran_scores[np.newaxis, :] / (distances_km + k)**2
    fpi_sums = np.sum(np.where(mask, fpi_terms, 0.0), axis=1)
    indie_df[f'fpi_{r}m'] = fpi_sums

3. 반경별 FPI 계산 및 변수 결합 진행 중...


---
# STEP5-3 원본 데이터에 결합

In [8]:
# 원본 데이터에 결합
final_df = pd.merge(df, indie_df[['business_id'] + [f'fpi_{r}m' for r in radii]], on='business_id', how='left')
for r in radii:
    final_df[f'fpi_{r}m'] = final_df[f'fpi_{r}m'].fillna(0)

final_df.to_csv(f"{PATH_to_save}/biz_sentiment_with_fpi.csv", index=False, encoding='utf-8-sig')
print(" 'biz_sentiment_with_fpi.csv' 저장 완료!")

 'biz_sentiment_with_fpi.csv' 저장 완료!


---
# step5-4 민감도분석

### 결론
본 프로젝트의 공간적 임계거리(Threshold Distance)는 '$100\text{m}$'로 제안함. 프랜차이즈의 실질적이고 유의미한 직접 타격권은 **$100\text{m}$**이며, 해당 영향력이 급격한 변곡점을 맞이하고 안정화되기 시작하는 경계선은 **$200\text{m}$**로 해석할 수 있음.

### 근거
1. 영향력의 크기 및 유의성: 
    프랜차이즈 밀집도($\text{FPI}$)가 독립 매장의 별점과 감성 점수 모두에 가장 강력하고 통계적으로 확연하게 유의미한 영향($p < 0.05$)을 미치는 범위는 $100\text{m}$ 반경이 유일함.
2. 계수의 안정화(수렴) 지점: 
    $100\text{m}$ 내에서 정점을 찍은 회귀계수(영향력)는 $200\text{m}$ 지점을 기점으로 급격히 둔화(반 토막 이하)되며, 이후 $300\text{m}\sim500\text{m}$ 구간까지 추가적인 변화 없이 일정한 수준($0.11 \sim 0.13$ 및 $0.0015$)으로 수렴하는 양상을 보임.

In [9]:
# 회귀분석 리포트 출력
for label, y_var in [("별점(Stars)", "stars"), ("감성점수(Sentiment)", "tfidf_sentiment")]:
    print("\n" + "="*60)
    print(f"     [민감도 분석] 독립 매장 {label}에 미치는 영향")
    print("="*60)
    summary_rows = []
    for r in radii:
        X = indie_df[f'fpi_{r}m']
        X = sm.add_constant(X)
        y = indie_df[y_var]
        model = sm.OLS(y, X).fit()
        summary_rows.append({
            '반경': f'{r}m',
            '회귀계수(Coef)': round(model.params.iloc[1], 4),
            'P-value': round(model.pvalues.iloc[1], 4),
            '설명력(R2)': round(model.rsquared, 4)
        })
    print(pd.DataFrame(summary_rows).to_string(index=False))


     [민감도 분석] 독립 매장 별점(Stars)에 미치는 영향
  반경  회귀계수(Coef)  P-value  설명력(R2)
100m      0.2880   0.0028   0.0018
200m      0.1069   0.0973   0.0006
300m      0.1303   0.0138   0.0013
500m      0.1135   0.0166   0.0012

     [민감도 분석] 독립 매장 감성점수(Sentiment)에 미치는 영향
  반경  회귀계수(Coef)  P-value  설명력(R2)
100m      0.0038   0.0412   0.0009
200m      0.0016   0.2040   0.0003
300m      0.0015   0.1286   0.0005
500m      0.0015   0.0916   0.0006
